## Get the price of ribs from Kroger


In [1]:
import os
import requests
from dotenv import load_dotenv

load_dotenv()

KROGER_API_BASE_URL = os.getenv("KROGER_API_BASE_URL_PROD")
KROGER_AUTH_TOKEN = None


def get_kroger_auth_token():
    auth_url = f"{KROGER_API_BASE_URL}/connect/oauth2/token"
    client_id = os.getenv("KROGER_CLIENT_ID_PROD")
    client_secret = os.getenv("KROGER_CLIENT_SECRET_PROD")
    auth = requests.auth.HTTPBasicAuth(client_id, client_secret)
    data = {"grant_type": "client_credentials", "scope": "product.compact"}

    response = requests.post(auth_url, auth=auth, data=data)

    if response.status_code == 200:
        KROGER_AUTH_TOKEN = response.json()["access_token"]
        return KROGER_AUTH_TOKEN
    else:
        print(f"Error obtaining auth token: {response.status_code} - {response.text}")
        return None


## Get auth token if not already set
KROGER_AUTH_TOKEN = get_kroger_auth_token()

### Get the list of Krogers Near Me


In [2]:
def get_kroger_locations(
    zipcode=None, radius=None, limit=None, latlong=None, auth_token=None
):

    filters = []

    if zipcode:
        filters.append(f"filter.zipCode.near={zipcode}")
    if radius:
        filters.append(f"filter.radiusInMiles={radius}")
    if limit:
        filters.append(f"filter.limit={limit}")
    if latlong:
        filters.append(f"filter.latLong.near={latlong}")

    filter_string = "&".join(filters)

    url = f"{KROGER_API_BASE_URL}/locations?{filter_string}"

    print(f"Fetching locations with URL: {url}")

    headers = {
        "Authorization": f"Bearer {auth_token}",
        "Content-Type": "application/json",
    }

    response = requests.get(url, headers=headers)

    if response.status_code == 200:
        return response.json()
    else:
        print(f"Error fetching locations: {response.status_code} - {response.text}")
        return None


locations = get_kroger_locations(
    latlong="38.2886059,-85.5628449", radius=7, auth_token=KROGER_AUTH_TOKEN
)

Fetching locations with URL: https://api.kroger.com/v1/locations?filter.radiusInMiles=7&filter.latLong.near=38.2886059,-85.5628449


In [ ]:
locations["data"][0]

{'locationId': '02400764',
 'storeNumber': '00764',
 'divisionNumber': '024',
 'chain': 'KROGER',
 'address': {'addressLine1': '9501 Westport Rd',
  'city': 'Louisville',
  'state': 'KY',
  'zipCode': '40241',
  'county': 'JEFFERSON COUNTY'},
 'geolocation': {'latitude': 38.2883838,
  'longitude': -85.5745714,
  'latLng': '38.2883838,-85.5745714'},
 'name': 'Kroger - Westport Plaza',
 'hours': {'timezone': 'America/New_York',
  'gmtOffset': '(UTC-05:00) Eastern Time (US Canada)',
  'open24': False,
  'monday': {'open': '06:00', 'close': '23:00', 'open24': False},
  'tuesday': {'open': '06:00', 'close': '23:00', 'open24': False},
  'wednesday': {'open': '06:00', 'close': '23:00', 'open24': False},
  'thursday': {'open': '06:00', 'close': '23:00', 'open24': False},
  'friday': {'open': '06:00', 'close': '23:00', 'open24': False},
  'saturday': {'open': '06:00', 'close': '23:00', 'open24': False},
  'sunday': {'open': '06:00', 'close': '23:00', 'open24': False}},
 'phone': '5024250065',
 

In [14]:
import json
import re


def create_kroger_location_url(location: dict) -> str:
    state = location["address"]["state"].lower()
    city = location["address"]["city"].lower()
    store_name = (
        re.sub(r"^'?Kroger\s*-\s*|'+$", "", location["name"])
        .strip()
        .lower()
        .replace(" ", "-")
    )
    division = location["divisionNumber"].lower()
    store_number = location["storeNumber"]

    result = f"https://www.kroger.com/stores/grocery/{state}/{city}/{store_name}/{division}/{store_number}/"

    return result


my_locations = [
    {
        "location_name": location["name"],
        "retailer": location["chain"],
        "retailer_store_id": location["storeNumber"],
        "address_street": location["address"]["addressLine1"],
        "address_city": location["address"]["city"],
        "address_state": location["address"]["state"],
        "address_zip": location["address"]["zipCode"],
        "location-url": create_kroger_location_url(location),
    }
    for location in locations["data"]
]

with open("kroger_location_ids.json", "w") as f:
    f.write(
        json.dumps(
            my_locations,
            indent=4,
        )
    )

In [15]:
import pandas as pd

df_locations = pd.DataFrame(my_locations)
df_locations.to_csv("kroger_locations_flat.csv", index=False)
df_locations.head()

,location_name,retailer,retailer_store_id,address_street,address_city,address_state,address_zip,location-url
0,Kroger - Westport Plaza,KROGER,00764,9501 Westport Rd,Louisville,KY,40241,https://www.kroger.com/stores/grocery/ky/louis...
1,Kroger - Springhurst,KROGER,00707,9440 Brownsboro Rd,Louisville,KY,40241,https://www.kroger.com/stores/grocery/ky/louis...
2,Kroger - Ballardsville Kroger,KROGER,00502,10010 Ballardsville Rd,Louisville,KY,40241,https://www.kroger.com/stores/grocery/ky/louis...
3,Kroger - Shops of Forest Springs,KROGER,00739,12450 La Grange Rd,Louisville,KY,40245,https://www.kroger.com/stores/grocery/ky/louis...
4,Kroger - Holiday Manor Shopping Center,KROGER,00186,2219 Holiday Manor Ctr,Louisville,KY,40222,https://www.kroger.com/stores/grocery/ky/louis...


## Search for Products


In [29]:
def kroger_product_search(
    search_term,
    auth_token,
    location_id=None,
    product_id=None,
    brand=None,
    fulfillment=None,
    limit=None,
):

    filters = []
    if location_id:
        filters.append(f"filter.locationId={location_id}")
    if product_id:
        filters.append(f"filter.productId={product_id}")
    if brand:
        filters.append(f"filter.brand={brand}")
    if fulfillment:
        filters.append(f"filter.fulfillment={fulfillment}")
    if limit:
        filters.append(f"filter.limit={limit}")

    filter_string = "&".join(filters)

    url = f"{KROGER_API_BASE_URL}/products?filter.term={search_term}&{filter_string}"

    print(f"Fetching products with URL: {url}")

    headers = {
        "Authorization": f"Bearer {auth_token}",
        "Content-Type": "application/json",
    }

    response = requests.get(url, headers=headers)

    if response.status_code == 200:
        return response.json()
    else:
        print(f"Error fetching products: {response.status_code} - {response.text}")
        return None


products = kroger_product_search(
    search_term="pork ribs",
    location_id="02400764",  # Westport Plaza
    auth_token=KROGER_AUTH_TOKEN,
)

print(
    len(
        [
            product
            for product in products["data"]
            if "Meat & Seafood" in product["categories"]
        ]
    )
)

Fetching products with URL: https://api.kroger.com/v1/products?filter.term=pork ribs&filter.locationId=02400764
7


In [30]:
def get_kroger_product_listings(UPC, location_id=None, auth_token=None):

    filters = []

    if location_id:
        filters.append(f"filter.locationId={location_id}")

    filter_string = "&".join(filters)

    url = f"{KROGER_API_BASE_URL}/products/{UPC}?{filter_string}"

    print(f"Fetching product listings with URL: {url}")

    headers = {
        "Authorization": f"Bearer {auth_token}",
        "Content-Type": "application/json",
    }

    response = requests.get(url, headers=headers)
    if response.status_code == 200:
        return response.json()
    else:
        print(
            f"Error fetching product listings: {response.status_code} - {response.text}"
        )


product_listing = get_kroger_product_listings(
    UPC="0020324300000", location_id="02400764", auth_token=KROGER_AUTH_TOKEN
)

Fetching product listings with URL: https://api.kroger.com/v1/products/0020324300000?filter.locationId=02400764


In [34]:
with open("kroger_location_ids.json", "r") as f:
    location_list = json.load(f)

location_list = [location["location_id"] for location in location_list]

product_listings = []

for location_id in location_list:
    product_listing = get_kroger_product_listings(
        UPC="0020324300000", location_id=location_id, auth_token=KROGER_AUTH_TOKEN
    )
    if product_listing:
        product_listings.append(
            {"location_id": location_id, "products": product_listing["data"]}
        )

Fetching product listings with URL: https://api.kroger.com/v1/products/0020324300000?filter.locationId=02400764
Fetching product listings with URL: https://api.kroger.com/v1/products/0020324300000?filter.locationId=02400707
Fetching product listings with URL: https://api.kroger.com/v1/products/0020324300000?filter.locationId=02400502
Fetching product listings with URL: https://api.kroger.com/v1/products/0020324300000?filter.locationId=02400739
Fetching product listings with URL: https://api.kroger.com/v1/products/0020324300000?filter.locationId=02400186
Fetching product listings with URL: https://api.kroger.com/v1/products/0020324300000?filter.locationId=02400356
Fetching product listings with URL: https://api.kroger.com/v1/products/0020324300000?filter.locationId=02400309
Fetching product listings with URL: https://api.kroger.com/v1/products/0020324300000?filter.locationId=02400387
Fetching product listings with URL: https://api.kroger.com/v1/products/0020324300000?filter.locationId=0

In [ ]:
product_listings[0]["products"]

{'productId': '0020324300000',
 'upc': '0020324300000',
 'productPageURI': '/p/pork-back-ribs/0020324300000?cid=dis.api.tpi_products-api_20240521_b:all_c:p_t:bbqplannerprod-bbc9l',
 'aisleLocations': [{'bayNumber': '13',
   'description': 'MEAT',
   'number': '101',
   'numberOfFacings': '1',
   'side': 'L',
   'shelfNumber': '1',
   'shelfPositionInBay': '3'},
  {'bayNumber': '13',
   'description': 'MEAT',
   'number': '101',
   'numberOfFacings': '1',
   'side': 'L',
   'shelfNumber': '2',
   'shelfPositionInBay': '3'}],
 'categories': ['Meat & Seafood', 'Natural & Organic'],
 'countryOrigin': 'UNITED STATES',
 'description': 'Pork Back Ribs',
 'snapEligible': True,
 'manufacturerDeclarations': ['Dairy Free',
  'Egg Free',
  'Fish Free',
  'Free from Peanuts',
  'Tree Nuts Free',
  'Soy Free',
  'Shellfish Free'],
 'allergens': [{'levelOfContainmentName': 'Free from',
   'name': 'Eggs and their derivates'},
  {'levelOfContainmentName': 'Free from',
   'name': 'Crustaceans and their 